In [ ]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

In [ ]:
import pandas as pd

# df1 = pd.read_csv('/content/drive/MyDrive/HIRA_NBD/01_csv/EDU200.csv', encoding='cp949')
# df2 = pd.read_csv('/content/drive/MyDrive/HIRA_NBD/01_csv/EDU400.csv', encoding='cp949')
# df3 = pd.read_csv('/content/drive/MyDrive/HIRA_NBD/01_csv/EDU530.csv', encoding='cp949')
# df4 = pd.read_csv('/content/drive/MyDrive/HIRA_NBD/01_csv/EDU300.csv', encoding='cp949')

# edu200 = df1.copy()
# edu400 = df2.copy()
# edu530 = df3.copy()
# edu300 = df4.copy()

import pandas as pd 

df1 = pd.read_csv('/Users/qwergs/Desktop/hira/EDU200.csv', encoding='cp949')
df2 = pd.read_csv('/Users/qwergs/Desktop/hira/EDU400.csv', encoding='cp949')
df3 = pd.read_csv('/Users/qwergs/Desktop/hira/EDU530.csv', encoding='cp949')
df4 = pd.read_csv('/Users/qwergs/Desktop/hira/EDU300.csv', encoding='cp949')

edu200 = df1.copy()
edu400 = df2.copy()
edu530 = df3.copy()
edu300 = df4.copy()

In [ ]:
import pandas as pd

# 1) 세부 호흡기 질환 매핑 딕셔너리(접두어 매칭)
resp_disease_map = {
    "호흡기질환": ["J",'A15','A16','A19',"S270",'P251',"A420","A430","B583","B664"],
    "감기": ["J00","J01","J02","J03","J04","J05","J06"],
    "인플루엔자": ["J09", "J10", "J11"],
    "결핵": ["A15", "A16","A19"],
    "만성폐쇠질환(COPD)": ["J431", "J432","J438","J439","J40","J41","J42","J43","J44","J47"],
    "천식": ["J45","J46"],
    "폐렴": ["A420","A430","B583","B664",
    "J120","J121","J122","J128","J1280","J1288","J129",
    "J13","J14","J150","J151","J152","J153","J154","J155",
    "J156","J157","J158","J159","J160","J168","J170","J171",
    "J173","J178","J180","J181","J188","J189",
    "J690","J691","J698","J853"],
    "기흉": ["J93", "S270", "P251"]
}

def extract_respiratory_and_categories(
    df: pd.DataFrame,
    main_col: str = "주상병코드",
    sub_col: str = "부상병코드",
    disease_map: dict = resp_disease_map,
    include_tb_in_respiratory: bool = False,   # 결핵(A코드)을 전체 호흡기에 포함할지 옵션
    include_nonJ_for_pneumothorax: bool = False  # 기흉 S/P코드 포함 여부 옵션
):
    """
    1) 전체 호흡기질환 환자(주상/부상 중 하나라도 J로 시작) 추출
    2) 그 안에서 매핑 리스트(prefix)로 각 세부 질환 데이터프레임 반환
       - 옵션으로 결핵(A15,A16,A19)이나 기흉의 비-J코드(S270,P251)를 전체 호흡기에 포함할 수 있음
    """
    df = df.copy()

    # 코드 문자열 정규화
    for c in (main_col, sub_col):
        df[c] = df[c].astype(str).str.upper().str.strip()

    # (기본) 전체 호흡기질환: J로 시작하는 코드가 주/부상병 중 하나라도 있으면 포함
    resp_mask = df[main_col].str.startswith("J", na=False) | df[sub_col].str.startswith("J", na=False)

    # 결핵(A코드)을 전체 호흡기에 포함하고 싶다면 추가
    if include_tb_in_respiratory and "결핵" in disease_map:
        tb_prefixes = tuple(disease_map["결핵"])
        tb_mask = df[main_col].str.startswith(tb_prefixes, na=False) | df[sub_col].str.startswith(tb_prefixes, na=False)
        resp_mask = resp_mask | tb_mask

    # 기흉의 비-J코드(S270, P251)까지 포함하고 싶다면 추가
    if include_nonJ_for_pneumothorax and "기흉" in disease_map:
        pneumo_extra = [p for p in disease_map["기흉"] if not p.startswith("J")]
        if pneumo_extra:
            px = tuple(pneumo_extra)
            px_mask = df[main_col].str.startswith(px, na=False) | df[sub_col].str.startswith(px, na=False)
            resp_mask = resp_mask | px_mask

    respiratory_df = df[resp_mask].copy()

    # 세부 질환별 부분집합 생성 (prefix 매칭)
    category_dfs = {}
    for name, prefixes in disease_map.items():
        pref = tuple(prefixes)
        m = respiratory_df[main_col].str.startswith(pref, na=False) | respiratory_df[sub_col].str.startswith(pref, na=False)
        category_dfs[name] = respiratory_df[m].copy()

    return respiratory_df, category_dfs

# -------------------- 사용 예시 --------------------
# 전체 호흡기질환(J*) + (필요시) 결핵/기흉 비-J코드 포함 여부 선택
respiratory_df, cat_dfs = extract_respiratory_and_categories(
    df1,
    main_col="주상병코드",
    sub_col="부상병코드",
    disease_map=resp_disease_map,
    include_tb_in_respiratory=True,          # 결핵을 전체 호흡기에 포함하려면 True
    include_nonJ_for_pneumothorax=True       # 기흉의 S/P 코드까지 포함하려면 True
)

# 각 세부 질환 데이터셋 접근
all_df       = cat_dfs['호흡기질환']
influenza_df = cat_dfs["인플루엔자"]
cold_df      = cat_dfs["감기"]
tb_df        = cat_dfs["결핵"]
copd_df      = cat_dfs["만성폐쇠질환(COPD)"]
asthma_df    = cat_dfs['천식']
pneumonia_df = cat_dfs["폐렴"]
pneumo_df    = cat_dfs["기흉"]

# 간단 요약
print("전체 호흡기질환 환자 수:", respiratory_df.shape[0])
for k, v in cat_dfs.items():
    print(f"{k} 환자 수:", v.shape[0])

In [ ]:
display(pneumonia_df)

# 재입원여부 컬럼 생성

In [ ]:
import numpy as np

target_patient_ids = [129, 972, 993, 1617, 2357, 2618, 3270, 3720]

# Ensuring the '수진자고유번호' column is of a comparable type (e.g., string or integer)
# Based on the info() output, it's an object type, likely strings or mixed types.
# Let's convert it to string for robust comparison.
pneumonia_df['수진자고유번호_str'] = pneumonia_df['수진자고유번호'].astype(str)
target_patient_ids_str = [str(id) for id in target_patient_ids]

pneumonia_df['재입원여부'] = np.where(
    pneumonia_df['수진자고유번호_str'].isin(target_patient_ids_str),
    1,
    0
)

# Drop the temporary string column if not needed
pneumonia_df = pneumonia_df.drop(columns=['수진자고유번호_str'])

display(pneumonia_df)

In [ ]:
# '재입원여부' 컬럼의 값 분포 확인
rehospitalization_counts = pneumonia_df['재입원여부'].value_counts()
print("Value counts for '재입원여부' column:")
print(rehospitalization_counts)

# 비율도 함께 확인
rehospitalization_ratios = pneumonia_df['재입원여부'].value_counts(normalize=True)
print("\nValue ratios for '재입원여부' column:")
print(rehospitalization_ratios)

In [ ]:
# '재입원여부'가 1인 행 필터링
rehospitalized_patients_df = pneumonia_df[pneumonia_df['재입원여부'] == 1]

# 필터링된 데이터프레임에서 '수진자고유번호'의 고유값 개수 확인
unique_rehospitalized_patients_count = rehospitalized_patients_df['수진자고유번호'].nunique()

print(f"'재입원여부'가 1인 고유한 수진자고유번호 개수: {unique_rehospitalized_patients_count}")

# 입원여부 컬럼 생성

In [ ]:
import numpy as np

# 새로운 target_patient_ids 리스트 정의
target_patient_ids_for_hospitalization = ['129', '135', '162', '202', '228', '229', '246', '287', '302',
'355', '471', '511', '599', '674', '723', '779', '833', '886',
'909', '949', '951', '972', '993', '1073', '1110', '1144', '1169',
'1275', '1306', '1471', '1481', '1504', '1510', '1535', '1565',
'1608', '1617', '1726', '1863', '2036', '2261', '2357', '2401',
'2417', '2476', '2492', '2603', '2618', '2746', '2779', '2808',
'2827', '2854', '2882', '2910', '3047', '3068', '3080', '3098',
'3154', '3270', '3371', '3406', '3415', '3452', '3462', '3491',
'3568', '3571', '3615', '3621', '3629', '3720', '3844']

# '수진자고유번호' 컬럼을 문자열 타입으로 변환 (일관성을 위해)
# 이전에 이미 문자열 컬럼을 사용했다면 불필요할 수 있지만 안전하게 다시 변환합니다.
pneumonia_df['수진자고유번호_str'] = pneumonia_df['수진자고유번호'].astype(str)


# '입원여부' 컬럼 생성 및 값 할당
pneumonia_df['입원여부'] = np.where(
    pneumonia_df['수진자고유번호_str'].isin(target_patient_ids_for_hospitalization),
    1,
    0
)

# 임시 문자열 컬럼 삭제 (필요 없는 경우)
pneumonia_df = pneumonia_df.drop(columns=['수진자고유번호_str'])

display(pneumonia_df.head())

In [ ]:
pneumonia_df.info()

In [ ]:
# '입원여부' 컬럼의 값 분포 확인
hospitalization_counts = pneumonia_df['입원여부'].value_counts()
print("Value counts for '입원여부' column:")
print(hospitalization_counts)

# 비율도 함께 확인
hospitalization_ratios = pneumonia_df['입원여부'].value_counts(normalize=True)
print("\nValue ratios for '입원여부' column:")
print(hospitalization_ratios)

In [ ]:
df_counts = pd.read_csv('/Users/qwergs/Desktop/df_counts.csv')

In [ ]:
df_counts.info()

In [ ]:
df_counts['성별'].value_counts()
# 1 : 남자 , 2 : 여자 

In [ ]:
df_counts['연령대'].value_counts()

In [ ]:
edu200_1 = edu200.copy()

In [ ]:
# 모든 컬럼을 숫자형으로 변환 (가능한 컬럼만 변환, 불가능한 값은 그대로 보존)
df_counts = df_counts.apply(pd.to_numeric, errors='ignore')
pneumonia_df = pneumonia_df.apply(pd.to_numeric, errors='ignore')
edu200_1 = edu200_1.apply(pd.to_numeric, errors='ignore')

In [ ]:
# 1) 연령을 숫자형으로 변환 (문자 → 숫자, 변환 안되면 NaN 유지)
edu200_1['연령'] = pd.to_numeric(edu200_1['연령'], errors='coerce')

# 2) 10단위로 묶어서 '연령대' 컬럼 생성
edu200_1['연령대'] = (edu200_1['연령'] // 10 * 10).astype('Int64').astype(str) + "대"

# 결과 확인
print(edu200_1[['연령', '연령대']].head())

# 각 연령대별 개수 확인
print(edu200_1['연령대'].value_counts().sort_index())

In [ ]:
# 두 데이터프레임에서 수진자고유번호 기준으로 병합 (내부조인)
merged = pd.merge(
    df_counts,
    pneumonia_df,
    on="명세서고유번호",
    how="outer",
    suffixes=("", "_dup")
)

# 중복 제거 (수진자고유번호 기준으로 유일하게 유지)
merged_unique = merged.drop_duplicates(subset=['명세서고유번호'])

print(merged_unique.shape)
print(merged_unique.head())

In [ ]:
merged.head()

In [ ]:
merged.shape

In [ ]:
merged.columns

In [ ]:
# from google.colab import files

# # 1) 엑셀 파일로 저장
# merged.to_excel("merged_result.xlsx", index=False)

# # 2) 파일 다운로드
# files.download("merged_result.xlsx")

In [ ]:
merged_expanded = pd.merge(
    merged, edu400, on='명세서고유번호', how='left'
)

In [ ]:
merged_expanded.columns

In [ ]:
merged_expanded.head(20)

In [ ]:
# from google.colab import files

# # 1) 엑셀 파일로 저장
# merged_expanded.to_excel("merged_expanded_result.xlsx", index=False)

# # 2) 파일 다운로드
# files.download("merged_expanded_result.xlsx")

In [ ]:
edu300.columns

In [ ]:
# 1) 영문→한글 매핑 딕셔너리
colmap = {
    "MID": "명세서고유번호",
    "JID": "수진자고유번호",
    "CZ_ITEM_CD": "항목코드",
    "DIV_TY_CD": "분류코드구분",
    "DIV_CD": "분류코드",
    "FQ1_MDCT_QTY": "1회투약량",
    "DY1_MDCT_QTY": "1일투약량",
    "DY1_INJC_QTY_EXEC_FQ": "1일투여량 또는 실시횟수",
    "TOT_INJC_DDCNT_EXEC_FQ": "총투여일수 또는 실시횟수",
    "TOT_USE_QTY_OR_EXEC_FQ": "총사용량 또는 실시횟수",
    "UNPRC": "단가",
    "AMT": "금액",
    "ADDC_ADT_AMT": "가산적용금액",
    "MEFT_DIV_NO": "약효분류번호",
    "EXP_TP_CD": "진료예외구분",
}

# 2) 실제 컬럼 이름 변경
def rename_to_korean(df: pd.DataFrame) -> pd.DataFrame:
    renamed = df.rename(columns=colmap)
    # 변경되지 않은(매핑에 없는) 영문 컬럼 확인 (참고용)
    not_mapped = [c for c in df.columns if c in colmap and colmap[c] == c]
    # 혹시 매핑 키가 있는데 원본에 없던 경우도 알려줌
    missing_in_df = [k for k in colmap.keys() if k not in df.columns]
    if missing_in_df:
        print("원본에 없던 영문 컬럼(참고):", missing_in_df)
    return renamed

# 사용 예시
# df_kor = rename_to_korean(df)
# print(df_kor.columns.tolist())

In [ ]:
edu300 = rename_to_korean(edu300)

In [ ]:
merged1 = pd.merge(
    merged, edu300, on='명세서고유번호', how='left'
)

In [ ]:
merged1.head(20)

In [ ]:
merged1.shape

In [ ]:
same_mask = merged_expanded['주상병코드'] == merged_expanded['상병코드']
same_mask.value_counts()

In [ ]:
# from google.colab import files

# # 1) 엑셀 파일로 저장
# merged1.to_excel("merged200_400.xlsx", index=False)

# # 2) 파일 다운로드
# files.download("merged200_400.xlsx")

In [ ]:
merged_expanded.info()

In [ ]:
df_no_na = merged_expanded.dropna(subset=['상병코드'])

print(df_no_na.shape)   # 제거 후 행 개수 확인
print(df_no_na.head())

In [ ]:
display(df_no_na['상병코드'].unique())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1) 필요한 컬럼만 선택
cols = ['성별', '연령대', '입내원일수', '진료과목', '요양급여비용총액', '상병코드', '수술여부']
df_cluster = merged_expanded[cols].copy()

# 2) 숫자형 처리
df_cluster['입내원일수'] = pd.to_numeric(df_cluster['입내원일수'], errors='coerce').fillna(0)
df_cluster['요양급여비용총액'] = pd.to_numeric(df_cluster['요양급여비용총액'], errors='coerce').fillna(0)

# 극단값이 많다면 로그 변환
df_cluster['요양급여비용총액'] = np.log1p(df_cluster['요양급여비용총액'])

# 3) 범주형 → 원핫인코딩
cat_cols = ['성별', '연령대', '진료과목', '상병코드', '수술여부']
df_encoded = pd.get_dummies(df_cluster, columns=cat_cols, drop_first=True)

# 4) 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)

In [ ]:
import hdbscan

# HDBSCAN 모델 생성
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, metric='euclidean')

# 군집 학습
cluster_labels = clusterer.fit_predict(X_scaled)

# 결과 저장
df_clustered = df_cluster.copy()
df_clustered['cluster'] = cluster_labels

print(df_clustered['cluster'].value_counts())

In [ ]:
summary = df_clustered.groupby('cluster').agg({
    '연령대': lambda x: x.value_counts().index[0],
    '진료과목': lambda x: x.value_counts().index[0],
    '요양급여비용총액': 'mean',
    '입내원일수': 'mean'
})
print(summary)

In [ ]:
# pip install umap-learn hdbscan

In [ ]:
import umap
import matplotlib.pyplot as plt
import seaborn as sns
import hdbscan

# ---------- 1) UMAP 차원 축소 ----------
umap_model = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=42)
X_umap = umap_model.fit_transform(X_scaled)

# ---------- 2) HDBSCAN ----------
clusterer = hdbscan.HDBSCAN(min_cluster_size=20, metric='euclidean')
cluster_labels = clusterer.fit_predict(X_scaled)

# ---------- 3) 결과 DataFrame ----------
df_clustered = df_cluster.copy()
df_clustered['cluster'] = cluster_labels
df_clustered['umap1'] = X_umap[:, 0]
df_clustered['umap2'] = X_umap[:, 1]

# ---------- 4) 시각화 ----------
plt.figure(figsize=(10, 7))
palette = sns.color_palette("tab10", len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0))
sns.scatterplot(
    data=df_clustered, x="umap1", y="umap2",
    hue="cluster", palette=palette, s=50, alpha=0.8
)
plt.title("HDBSCAN Clusters (UMAP Projection)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
import hdbscan
import umap
import matplotlib.pyplot as plt

# X_scaled: 전처리/원핫 후 스케일된 특성 행렬이라고 가정
# (모든-0 행은 코사인 거리 불가 → 제거)
row_norm = np.linalg.norm(X_scaled, axis=1)
valid_idx = row_norm > 0
X_valid = X_scaled[valid_idx]

# 1) 코사인 거리 행렬 계산
D = pairwise_distances(X_valid, metric='cosine')

# 2) HDBSCAN (사전계산 거리 사용)
clusterer = hdbscan.HDBSCAN(min_cluster_size=20, metric='precomputed')
labels_valid = clusterer.fit_predict(D)

# 3) 원본 인덱스에 라벨 복원
labels = np.full(X_scaled.shape[0], -1)    # 기본 노이즈로 채움
labels[valid_idx] = labels_valid

# 4) UMAP 시각화도 코사인 기준 추천
umap_model = umap.UMAP(n_neighbors=30, min_dist=0.1, metric='cosine', random_state=42)
X_umap = umap_model.fit_transform(X_scaled)

# 5) 그림
plt.figure(figsize=(9,7))
scatter = plt.scatter(X_umap[:,0], X_umap[:,1], c=labels, s=18, alpha=0.8)
plt.title("HDBSCAN (cosine) on UMAP(cosine)")
plt.colorbar(scatter, label='cluster (-1=noise)')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, normalize

use_cols = ['성별','연령대','입내원일수','진료과목','요양급여비용총액','상병코드','수술여부']
dfc = merged_expanded[use_cols].copy()

# 숫자형
dfc['입내원일수'] = pd.to_numeric(dfc['입내원일수'], errors='coerce').fillna(0)
dfc['요양급여비용총액'] = pd.to_numeric(dfc['요양급여비용총액'], errors='coerce').fillna(0)
# 비용은 로그 권장 (극단값 완화)
dfc['요양급여비용총액'] = np.log1p(dfc['요양급여비용총액'])

# 범주형 → 원핫
cat_cols = ['성별','연령대','진료과목','상병코드','수술여부']
X = pd.get_dummies(dfc, columns=cat_cols, drop_first=True)

# 스케일링(원핫 포함 전체를 표준화하면 수렴이 안정적)
scaler = StandardScaler(with_mean=False)  # 희소 행렬 호환을 위해 with_mean=False 권장
X_scaled = scaler.fit_transform(X)

# L2 정규화 (코사인 기반 효과)
X_unit = normalize(X_scaled, norm='l2', axis=1)

# 완전 0벡터 제거(드물지만 있을 수 있음)
row_norm = np.linalg.norm(X_unit, axis=1)
mask = row_norm > 0
X_unit = X_unit[mask]
dfc_valid = dfc.loc[mask].reset_index(drop=True)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# k 후보에서 실루엣으로 간단 탐색
sil = {}
for k in [3,4,5,6,7,8]:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_unit)
    sil[k] = silhouette_score(X_unit, labels, metric='euclidean')  # 단위벡터라 euclidean≈cosine

best_k = max(sil, key=sil.get)
km = KMeans(n_clusters=best_k, n_init=20, random_state=42)
labels = km.fit_predict(X_unit)

df_clustered = dfc_valid.copy()
df_clustered['cluster'] = labels
print("선택된 k:", best_k)
print(df_clustered['cluster'].value_counts())

In [ ]:
import umap
import matplotlib.pyplot as plt

um = umap.UMAP(n_neighbors=30, min_dist=0.1, metric='cosine', random_state=42)
emb = um.fit_transform(X_unit)

plt.figure(figsize=(9,7))
sc = plt.scatter(emb[:,0], emb[:,1], c=labels, s=18, alpha=0.8)
plt.title(f"K-Means (k={best_k}) on UMAP (cosine)")
plt.colorbar(sc, label='cluster')
plt.show()

In [ ]:
# 예: 평균 체류/비용 + 최빈 범주
summary = df_clustered.join(pd.DataFrame(X.index[mask])).groupby('cluster').agg({
    '입내원일수':'mean',
    '요양급여비용총액':'mean'
})
print(summary.round(2))

# 범주형 최빈값(간단 버전)
def mode_of(s):
    vc = s.value_counts()
    return vc.index[0] if len(vc) else np.nan

desc = df_clustered.groupby('cluster').agg({
    '성별': mode_of,
    '연령대': mode_of,
    '진료과목': mode_of,
    '상병코드': mode_of,
    '수술여부': mode_of
})
print(desc)

In [ ]:
grouped = (
    merged_expanded
    .groupby("명세서고유번호")
    .agg({
        "주상병코드": lambda x: list(x.dropna().unique()),
        "부상병코드": lambda x: list(x.dropna().unique()),
        "상병코드": lambda x: list(x.dropna().unique())
    })
    .reset_index()
)

print(grouped.head())

In [ ]:
display(grouped.head(20))

In [ ]:
# df_no_na
csv_path = '/Users/qwergs/Desktop/ICD-10-CSV-master/2020/diagnosis.csv'

In [ ]:
mapping = pd.read_csv(csv_path)

In [ ]:
mapping.columns

In [ ]:
df_no_na.columns

In [ ]:
import re

# 1) ICD-10 매핑 파일 읽기
icd = pd.read_csv('/Users/qwergs/Desktop/ICD-10-CSV-master/2020/diagnosis.csv')

# 2) 코드 전처리 함수 (점 제거, 대문자)
def norm_code(x):
    if pd.isna(x):
        return ""
    x = str(x).upper().strip()
    x = x.replace(".", "")
    x = re.sub(r"[^A-Z0-9]", "", x)
    return x

# 3) ICD 매핑 테이블 정리
icd["_CODE_N"] = icd["Code"].map(norm_code)
icd_map = dict(zip(icd["_CODE_N"], icd["LongDescription"].astype(str)))

# 4) df_no_na의 상병코드 정규화 + 매핑
df_no_na["상병명"] = (
    df_no_na["상병코드"]
    .astype(str)
    .map(norm_code)
    .map(icd_map)               # 매핑 수행
    .fillna("없음 (매핑 필요)")  # 못 찾으면 표시
)

# 결과 확인
print(df_no_na[["상병코드", "상병명"]].head(20))

In [ ]:
df_no_na.columns

In [ ]:
grouped = (
    df_no_na
    .groupby("명세서고유번호")
    .agg({
        "주상병코드": lambda x: list(x.dropna().unique()),
        "부상병코드": lambda x: list(x.dropna().unique()),
        "상병명": lambda x: list(x.dropna().unique())
    })
    .reset_index()
)

display(grouped.head(20))

In [ ]:
df_counts['연령대'].value_counts()